# 01 - Synthetic Epidemiological Data Generation

This notebook documents the generation of a synthetic spatio-temporal epidemiological dataset for early viral outbreak prediction.

The dataset is generated using a stochastic simulation that incorporates regional disease transmission, mobility, vaccination, climate variables, testing capacity, healthcare pressure and public health interventions.

## 1. Project Objective

The objective of this notebook is to generate a reproducible synthetic dataset that can be used to train machine learning models for predicting viral outbreak risk 7 days in advance.

The final prediction target is:

`outbreak_7d`

This variable indicates whether a given region is expected to experience outbreak risk within the next seven days.

## 2. Simulation Assumptions

The simulation is based on the following assumptions:

- Regions are connected through a spatial diffusion network.
- Transmission depends on Rt, mobility, vaccination and environmental conditions.
- Higher mobility increases imported cases and spatial spread.
- Vaccination reduces effective transmission.
- Public health interventions are activated when regional incidence becomes elevated.
- Reported cases differ from true cases due to imperfect testing and detection.
- The target variable is defined using future epidemiological risk indicators.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 3. Project Directory Check

Before running the simulation, we check that the notebook is being executed from the root folder of the repository.

In [ ]:
!pwd
!ls

## 4. Running the Simulation Script

The dataset is generated using the script:

`src/simulate_data.py`

This script creates the synthetic epidemiological dataset and saves it in:

`data/raw/epiwatch_epidemiological_surveillance_dataset.csv`

In [ ]:
!python src/simulate_data.py

## 5. Loading the Generated Dataset

In [ ]:
DATA_PATH = Path("data/raw/epiwatch_epidemiological_surveillance_dataset.csv")

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

df.head()

## 6. Dataset Dimensions

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

## 7. Dataset Structure

In [ ]:
df.info()

## 8. Main Variables

The dataset includes several groups of variables:

### Epidemiological variables
- `reported_cases`
- `true_cases`
- `Rt`
- `effective_R`
- `spatial_pressure`
- `imported_cases`
- `incidence_per_100k`

### Environmental variables
- `temperature`
- `humidity`
- `rainfall`

### Population and healthcare variables
- `population`
- `population_density`
- `testing_rate`
- `healthcare_capacity`
- `healthcare_pressure`

### Intervention and mobility variables
- `mobility_index`
- `vaccination_rate`
- `intervention_active`

### Prediction target
- `outbreak_7d`

In [ ]:
df.describe().T

## 9. Target Distribution

A useful outbreak prediction dataset should contain both outbreak and non-outbreak examples.

In [ ]:
target_distribution = df["outbreak_7d"].value_counts(normalize=True)
target_distribution

In [ ]:
plt.figure(figsize=(6, 4))
df["outbreak_7d"].value_counts().sort_index().plot(kind="bar")

plt.title("Distribution of 7-Day Outbreak Risk Classes")
plt.xlabel("Outbreak within 7 days")
plt.ylabel("Number of observations")
plt.xticks(
    ticks=[0, 1],
    labels=["No outbreak", "Outbreak"],
    rotation=0,
)

plt.tight_layout()
plt.show()

## 10. Temporal Coverage

In [ ]:
print("Start date:", df["date"].min())
print("End date:", df["date"].max())
print("Number of regions:", df["region"].nunique())

## 11. Reported Cases Over Time

In [ ]:
daily_cases = (
    df.groupby("date")["reported_cases"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(12, 5))
plt.plot(daily_cases["date"], daily_cases["reported_cases"])

plt.title("Total Reported Cases Over Time")
plt.xlabel("Date")
plt.ylabel("Reported cases")

plt.tight_layout()
plt.show()

## 12. Regional Disease Burden

In [ ]:
regional_cases = (
    df.groupby("region")["reported_cases"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
regional_cases.plot(kind="bar")

plt.title("Total Reported Cases by Region")
plt.xlabel("Region")
plt.ylabel("Total reported cases")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 13. Example Regional Time Series

In [ ]:
top_regions = regional_cases.head(5).index

plt.figure(figsize=(12, 6))

for region in top_regions:
    subset = df[df["region"] == region]
    plt.plot(subset["date"], subset["reported_cases"], label=region)

plt.title("Reported Cases Over Time in Highest-Burden Regions")
plt.xlabel("Date")
plt.ylabel("Reported cases")
plt.legend(title="Region")

plt.tight_layout()
plt.show()

## 14. Final Dataset Preview

In [ ]:
df.sample(10, random_state=42)

## 15. Conclusion

This notebook generated and inspected a synthetic spatio-temporal epidemiological dataset for viral outbreak prediction.

The dataset contains:

- 20 simulated regions
- daily observations
- epidemiological, environmental, mobility and healthcare-related variables
- a future prediction target: `outbreak_7d`

The next step is exploratory data analysis, where the relationships between variables and outbreak risk will be examined in greater detail.